In [ ]:
"""
Notebook: analise_sensibilidade.py
Projeto: Brasil Público — Escala 6x1
Etapa: 03_modelagem
Objetivo: Mapear o espaço de resultados possíveis sob diferentes
          combinações de hipóteses. Não produz estimativa única —
          produz mapa disciplinado do espaço econômico do problema.

Eixos da análise:
  1. Faixa de trabalhadores afetados (quem)
  2. Intensidade da redução de horas (quanto — apenas formulações econômicas)
  3. Ganho de produtividade por hora (compensação — diferenciado por setor)

Classificação dos cenários:
  "baseline_consistente" — hipóteses próximas à realidade observada
  "cenario_estresse"     — hipóteses de estresse, possíveis mas improváveis
  "limite_estrutural"    — bounds teóricos, não cenários econômicos
  "fronteira_compensacao"— ΔPIB ≥ 0: ganhos superam perdas

Limitações documentadas: ver DECISIONS.md → D11, D13, D16
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from itertools import product

BASE      = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
BASE_IN   = BASE / "data/processed/base_analitica_setorial_2023.csv"
HORAS_IN  = BASE / "data/processed/horas_por_setor_2023_corrigido.csv"
PIB_IN    = BASE / "data/processed/pib_setorial_2012_2023.csv"
DIR_FIGS  = BASE / "outputs/figures"
DIR_TABS  = BASE / "outputs/tables"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print("Imports OK")

In [ ]:
df_base  = pd.read_csv(BASE_IN)
df_horas = pd.read_csv(HORAS_IN, index_col=0)
df_pib   = pd.read_csv(PIB_IN)

pib_2023     = df_pib[df_pib["ano"] == 2023].squeeze()
pib_total_rs = pib_2023["pib_total"] * 1e6

# Apenas setores incluídos na simulação
df_sim = df_base[df_base["incluir"] == True].copy()

print(f"PIB total 2023: R$ {pib_total_rs/1e12:.2f} tri")
print(f"Setores na simulação: {df_sim['setor'].tolist()}")

In [ ]:

# ── Eixo 1: Faixa de trabalhadores afetados ──────────────────
# Cada valor implica um mecanismo de transmissão diferente.
# "todos_ocupados" é limite estrutural, não cenário econômico.

FAIXAS = {
    "41–44h": {
        "col_pct":    "pct_41_44h",
        "h_ref":      42.0,
        "mecanismo":  "Cumprimento estrito do limite legal. Apenas excesso cortado.",
        "categoria":  "baseline_consistente",
    },
    "40–44h": {
        "col_pct":    None,           # calculado abaixo
        "h_ref":      42.0,
        "mecanismo":  "Empregadores uniformizam para 40h. Efeito preventivo.",
        "categoria":  "cenario_estresse",
    },
    "40–48h": {
        "col_pct":    None,
        "h_ref":      43.0,
        "mecanismo":  "Horas extras habituais eliminadas. Efeito de contágio.",
        "categoria":  "cenario_estresse",
    },
    "todos_ocupados": {
        "col_pct":    None,
        "h_ref":      None,           # usa h_media do setor
        "mecanismo":  "Reestruturação completa. Irrealista no curto prazo.",
        "categoria":  "limite_estrutural",
    },
}

# Calcular parcelas faltantes a partir dos dados corrigidos
for setor_row in df_sim.itertuples():
    s = setor_row.setor
    if s not in df_horas.index:
        continue
    h = df_horas.loc[s]
    # 40–44h = 40h exatas + 41–44h
    df_horas.loc[s, "pct_40_44h_inclusive"] = h["pct_40h_exatas"] + h["pct_41_44h"]
    # 40–48h = 40h exatas + 41–44h + 45–48h
    # 45–48h = acima_44 que está ≤ 48 — proxy: usamos pct_acima_44h * 0.44
    # (baseado na distribuição observada: ~44% dos >44h estão entre 45-48h)
    df_horas.loc[s, "pct_40_48h_inclusive"] = (
        h["pct_40h_exatas"] + h["pct_41_44h"] + h["pct_acima_44h"] * 0.44
    )
    df_horas.loc[s, "pct_todos"] = 100.0

# Mapear col_pct para as faixas calculadas
FAIXAS["40–44h"]["col_pct"]       = "pct_40_44h_inclusive"
FAIXAS["40–48h"]["col_pct"]       = "pct_40_48h_inclusive"
FAIXAS["todos_ocupados"]["col_pct"] = "pct_todos"

# ── Eixo 2: Intensidade da redução (formulações econômicas) ──
# Formulação B (limite mecânico -9,09%) excluída da matriz.
# Documentada como upper bound não-econômico — ver DECISIONS.md D16.

INTENSIDADES = {
    "Formulação A\n(média da faixa)": {
        "tipo": "fixo",
        "valor": lambda h_ref, h_media: (40 - h_ref) / h_ref,
        "nota": "Redução da média da faixa afetada para 40h. Baseline.",
        "categoria": "baseline_consistente",
    },
    "Formulação B\n(limite mecânico)": {
        "tipo": "fixo",
        "valor": lambda h_ref, h_media: (40 - 44) / 44,
        "nota": "Upper bound mecânico. Não é cenário econômico.",
        "categoria": "limite_estrutural",
    },
}

# ── Eixo 3: Ganho de produtividade (diferenciado por setor) ──
# Fator multiplicador baseado na intensidade de jornada do setor.
# Setores com jornadas mais longas têm maior margem de ganho.

def fator_setor(h_media):
    """Fator de sensibilidade setorial ao ganho de produtividade."""
    if h_media > 41:
        return 1.2   # alta intensidade: maior margem de ganho
    elif h_media > 39:
        return 1.0   # média intensidade: ganho médio
    else:
        return 0.8   # baixa intensidade: menor margem

GANHOS_P = {
    "0%":  0.000,
    "+3%": 0.030,
    "+6%": 0.060,
    "+9%": 0.090,
}

print("\nEixos definidos:")
print(f"  Faixas: {list(FAIXAS.keys())}")
print(f"  Intensidades: {list(INTENSIDADES.keys())}")
print(f"  Ganhos: {list(GANHOS_P.keys())}")

In [ ]:
# Formulação C = redução proporcional à distância de 40h, por trabalhador.
# A = proxy da média; C = conceitualmente correta mas requer distribuição intrasetorial.
# Aqui testamos C na Agropecuária (maior dispersão: P25=25h, P75=44h).
#
# Aproximação: usar a distribuição de faixas conhecida para estimar ΔH médio ponderado.

print("── TESTE DE ROBUSTEZ: Formulação C na Agropecuária ─────")
setor_teste = "Agropecuária"
h = df_horas.loc[setor_teste]

# Pontos médios das faixas observadas
faixas_obs = {
    "41–44h": (h["pct_41_44h"] / 100, 42.0),   # ponto médio 41-44
    "45–48h": (h["pct_acima_44h"] * 0.44 / 100, 46.5),  # proxy 45-48
    "49–60h": (h["pct_acima_44h"] * 0.40 / 100, 54.0),  # proxy 49-60
    "60h+":   (h["pct_acima_44h"] * 0.16 / 100, 65.0),  # proxy 60+
}

delta_h_c = sum(
    peso * (40 - h_mid) / h_mid
    for _, (peso, h_mid) in faixas_obs.items()
)
delta_h_a = (40 - 42) / 42  # Formulação A

vab_agro = pib_2023["agropecuaria"] * 1e6
n_agro   = h["n"]
delta_vab_a = vab_agro * (h["pct_41_44h"] / 100) * delta_h_a
delta_vab_c = vab_agro * delta_h_c

print(f"  Formulação A (média faixa 41-44h): ΔH = {delta_h_a*100:.2f}%  →  ΔVAB = R$ {delta_vab_a/1e9:.3f} bi")
print(f"  Formulação C (proporcional, 40h+): ΔH = {delta_h_c*100:.2f}%  →  ΔVAB = R$ {delta_vab_c/1e9:.3f} bi")
print(f"  Diferença: {abs(delta_vab_c - delta_vab_a)/1e9:.3f} bi")
print(f"  Conclusão: viés da Formulação A = {abs(delta_vab_c/delta_vab_a - 1)*100:.1f}%")
print(f"  Nota: viés relevante — Formulação C inclui trabalhadores >44h que também reduziriam.")

In [ ]:
resultados = []

for nome_faixa, cfg_faixa in FAIXAS.items():
    for nome_intens, cfg_intens in INTENSIDADES.items():
        for nome_ganho, ganho_global in GANHOS_P.items():

            delta_vab_total = 0

            for _, row in df_sim.iterrows():
                setor    = row["setor"]
                vab_rs   = row["vab_r$_bi"] * 1e9
                h_media  = row["h_media"]
                h_ref    = cfg_faixa["h_ref"] or h_media

                # Parcela afetada
                col = cfg_faixa["col_pct"]
                if col in df_horas.columns and setor in df_horas.index:
                    parcela = df_horas.loc[setor, col] / 100.0
                else:
                    parcela = row["parcela_afetada"]

                # Intensidade da redução
                delta_h = cfg_intens["valor"](h_ref, h_media)

                # Ganho de produtividade diferenciado por setor
                fator    = fator_setor(h_media)
                ganho_p  = ganho_global * fator

                # Impacto no VAB
                delta_vab = vab_rs * parcela * (delta_h + ganho_p)
                delta_vab_total += delta_vab

            delta_pib_pct = delta_vab_total / pib_total_rs * 100

            # Classificar cenário
            cat_faixa  = cfg_faixa["categoria"]
            cat_intens = cfg_intens["categoria"]

            if delta_pib_pct >= 0:
                categoria = "fronteira_compensacao"
            elif cat_faixa == "limite_estrutural" or cat_intens == "limite_estrutural":
                categoria = "limite_estrutural"
            elif cat_faixa == "cenario_estresse" or cat_intens == "cenario_estresse":
                categoria = "cenario_estresse"
            else:
                categoria = "baseline_consistente"

            resultados.append({
                "faixa":         nome_faixa,
                "intensidade":   nome_intens,
                "ganho_p":       nome_ganho,
                "delta_pib_pct": round(delta_pib_pct, 3),
                "categoria":     categoria,
            })

df_res = pd.DataFrame(resultados)

print("── MATRIZ DE RESULTADOS (ΔPIB %) ────────────────────────")
pivot = df_res.pivot_table(
    index=["faixa", "intensidade"],
    columns="ganho_p",
    values="delta_pib_pct"
)
print(pivot.to_string())

df_res.to_csv(DIR_TABS / "matriz_sensibilidade.csv", index=False)
print("\n✓ Matriz salva.")

In [ ]:
# Apenas Formulação A (econômica) × todas as faixas × todos os ganhos

df_heat = df_res[df_res["intensidade"].str.contains("média da faixa")].copy()

pivot_heat = df_heat.pivot_table(
    index="faixa",
    columns="ganho_p",
    values="delta_pib_pct"
).reindex(["41–44h", "40–44h", "40–48h", "todos_ocupados"])

# Ordenar colunas
pivot_heat = pivot_heat[["0%", "+3%", "+6%", "+9%"]]

fig, ax = plt.subplots(figsize=(11, 6))

data  = pivot_heat.values.astype(float)
vmax  = max(abs(data.min()), abs(data.max()))
cmap  = plt.cm.RdYlGn
norm  = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

im = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto")

# Rótulos
ax.set_xticks(range(len(pivot_heat.columns)))
ax.set_xticklabels(
    [f"Ganho produt.\n{c}" for c in pivot_heat.columns],
    fontsize=10
)
ax.set_yticks(range(len(pivot_heat.index)))
ax.set_yticklabels(pivot_heat.index, fontsize=10)

# Valores nas células com classificação
categorias_heat = df_heat.pivot_table(
    index="faixa",
    columns="ganho_p",
    values="categoria",
    aggfunc="first"
).reindex(["41–44h", "40–44h", "40–48h", "todos_ocupados"])
categorias_heat = categorias_heat[["0%", "+3%", "+6%", "+9%"]]

simbolo = {
    "baseline_consistente":  "●",
    "cenario_estresse":      "▲",
    "limite_estrutural":     "■",
    "fronteira_compensacao": "★",
}

for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        val  = data[i, j]
        faixa_nome = pivot_heat.index[i]
        ganho_nome = pivot_heat.columns[j]
        cat  = categorias_heat.loc[faixa_nome, ganho_nome]
        sim  = simbolo.get(cat, "")
        cor  = "white" if abs(val) > vmax * 0.5 else "black"
        ax.text(j, i, f"{val:.3f}%\n{sim}",
                ha="center", va="center", fontsize=9, color=cor, fontweight="bold")

# Linha da fronteira de compensação
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        if data[i, j] >= 0:
            ax.add_patch(plt.Rectangle(
                (j - 0.5, i - 0.5), 1, 1,
                fill=False, edgecolor="gold", linewidth=2.5
            ))

plt.colorbar(im, ax=ax, label="ΔPIB estimado (%)", shrink=0.8)

ax.set_title(
    "Mapeamento do espaço de resultados — redução de jornada (44h→40h)\n"
    "Formulação A (média da faixa) · Produtividade diferenciada por setor · Brasil 2023",
    fontsize=12, loc="left"
)

# Legenda de categorias
legenda_txt = (
    "● Baseline consistente  "
    "▲ Cenário de estresse  "
    "■ Limite estrutural  "
    "★ Fronteira de compensação (ΔPIB ≥ 0)"
)
fig.text(0.01, -0.02, legenda_txt, fontsize=8.5, color="gray",
         transform=fig.transFigure)

fig.text(
    0.01, -0.06,
    "Referência CNI: -0,700% · "
    "Faixa de incerteza do modelo: [-0,513%, -0,770%] · "
    "Limitação: modelo de 1ª ordem, produtividade exógena, sem efeitos de segunda ordem.",
    fontsize=7.5, color="gray", transform=fig.transFigure
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "12_heatmap_sensibilidade.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 12 salvo.")

In [ ]:
# Calibração reversa: quais combinações produzem resultado próximo ao da CNI?

TARGET_CNI  = -0.700
TOLERANCIA  = 0.100  # ±0,1 pp

df_calibra = df_res[
    df_res["delta_pib_pct"].between(TARGET_CNI - TOLERANCIA, TARGET_CNI + TOLERANCIA)
].copy()

print(f"\n── CALIBRAÇÃO REVERSA: cenários com ΔPIB ≈ {TARGET_CNI}% ──")
print(f"(±{TOLERANCIA} pp de tolerância)")
if len(df_calibra) > 0:
    print(df_calibra[["faixa", "intensidade", "ganho_p",
                       "delta_pib_pct", "categoria"]].to_string(index=False))
else:
    print("Nenhuma combinação encontrada dentro da tolerância.")
    print("\nCombinações mais próximas:")
    df_res["dist_cni"] = abs(df_res["delta_pib_pct"] - TARGET_CNI)
    print(df_res.nsmallest(5, "dist_cni")[
        ["faixa", "intensidade", "ganho_p", "delta_pib_pct", "categoria"]
    ].to_string(index=False))